In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
from constants import EQ, HH, custom_tech_colors, custom_metal_colors

In [ ]:
# Damage files
df_sps_damage = pd.read_csv(r'results/df_sps_damage_regionalized.csv')
df_nze_damage = pd.read_csv(r'results/df_nze_damage_regionalized.csv')

In [ ]:
df_sps_damage['Metal'].unique()

In [ ]:
# Group into one category for plotting
df_sps_damage["Technology"] = df_sps_damage["Technology"].replace("Other low emissions power generation", "Low emissions power generation")
df_nze_damage["Technology"] = df_nze_damage["Technology"].replace("Other low emissions power generation", "Low emissions power generation")

In [ ]:
# Rename
df_sps_damage["Metal"] = df_sps_damage["Metal"].replace("Graphite (battery-grade)", "Graphite")
df_nze_damage["Metal"] = df_nze_damage["Metal"].replace("Graphite (battery-grade)", "Graphite")

In [ ]:
# Midpoints file
df_sps_mp = pd.read_csv(r'results/df_sps_midpoints_regionalized.csv')
df_nze_mp = pd.read_csv(r'results/df_nze_midpoints_regionalized.csv')

In [ ]:
# Additional impact file
df_comparison = pd.read_csv(r'results/comparaison_benefits_burden.csv')

In [ ]:
from matplotlib import rcParams

# Set global Matplotlib parameters
rcParams['pdf.fonttype'] = 42  # Ensure TrueType fonts are embedded
rcParams['ps.fonttype'] = 42
rcParams['font.family'] = 'Arial'  # Use serif fonts like Times New Roman or Palatino
rcParams['font.size'] = 10
rcParams['axes.labelsize'] = 10
rcParams['legend.fontsize'] = 9
rcParams['xtick.labelsize'] = 9
rcParams['ytick.labelsize'] = 9
rcParams['axes.titlesize'] = 12

# Stacked bar from 2022 to 2050 for metals and technologies

In [ ]:
def plot_stacked_area(df, group_by, value_col, title, y_label, color_palette="tab20", custom_colors=None, threshold=0.01, save_path="plots/stacked_area"):
    """
    Generate a stacked area plot with optional custom colors and conditional aggregation.

    Parameters:
    - df: DataFrame with data
    - group_by: Column to group by ('Metal' or 'Technology')
    - value_col: Column containing impact values
    - title: Plot title
    - y_label: Label for Y-axis (e.g., "Impact (unit)")
    - color_palette: Colormap name for automatic coloring (default: "tab20")
    - custom_colors: Dict with manual colors (e.g., {"Solar PV": "#1f77b4"})
    - threshold: Threshold for grouping small categories into "Other" (applies only to Metals, not Technologies)
    - save_path: Base filename for saving plots (without extension)
    
    Outputs:
    - Saves PDF and PNG versions of the figure
    """

    # Ensure directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Aggregate data
    df_grouped = df.groupby(["Year", group_by])[value_col].sum().reset_index()

    # Apply threshold only for Metals, keep all Technologies
    if group_by == "Metal":
        total_contributions = df_grouped.groupby(group_by)[value_col].sum()
        significant_categories = total_contributions[total_contributions / total_contributions.sum() >= threshold].index
        df_grouped[group_by] = df_grouped[group_by].apply(lambda x: x if x in significant_categories else "Other")

    # Pivot for stacked area plot
    df_pivot = df_grouped.pivot_table(index="Year", columns=group_by, values=value_col, aggfunc="sum")

    # Sort categories based on first year's contribution (largest to smallest)
    first_year = df_pivot.index.min()
    sorted_categories = df_pivot.loc[first_year].sort_values(ascending=False).index
    df_pivot = df_pivot[sorted_categories]

    # Assign colors (Fix: Apply to both Metals & Technologies)
    unique_categories = df_pivot.columns
    if custom_colors:
        # Apply user-defined colors for both Metals & Technologies
        color_dict = {cat: custom_colors.get(cat, "gray") for cat in unique_categories}
    else:
        # Otherwise, use colormap
        cmap = cm.get_cmap(color_palette, len(unique_categories))
        color_dict = {cat: cmap(i) for i, cat in enumerate(unique_categories)}

    # Generate plot
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    colors = [color_dict.get(col, "gray") for col in df_pivot.columns]
    ax.stackplot(df_pivot.index, df_pivot.T, labels=df_pivot.columns, colors=colors, alpha=0.8)

    # Formatting
    ax.set_title(title)
    ax.set_ylabel(y_label)

    # Sort legend in the same order as the stacked areas
    handles, labels = ax.get_legend_handles_labels()
    legend_order = [labels.index(cat) for cat in sorted_categories if cat in labels]
    sorted_handles = [handles[i] for i in legend_order]
    sorted_labels = [labels[i] for i in legend_order]

    ax.legend(sorted_handles, sorted_labels, loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()

    # Save figures
    plt.savefig(f"{save_path}.pdf", format="pdf", dpi=600, transparent=True)
    plt.savefig(f"{save_path}.png", format="png", dpi=600)
    plt.show()


In [ ]:
# Generate plots
fig_sps_metal_eq = plot_stacked_area(df_sps_damage, "Metal", "Total ecosystem quality", "Ecosystem Quality Damage by Metal", "PDF.m2.yr", custom_colors=custom_metal_colors, save_path="results/plots/stacked_area_metals_technologies/fig_sps_metal_eq")
fig_sps_metal_hh = plot_stacked_area(df_sps_damage, "Metal", "Total human health", "Human Health Damage by Metal", "DALY", custom_colors=custom_metal_colors, save_path="results/plots/stacked_area_metals_technologies/fig_sps_metal_hh")
fig_sps_tech_eq = plot_stacked_area(df_sps_damage, "Technology", "Total ecosystem quality", "Ecosystem Quality Damage by Technology", "PDF.m2.yr", custom_colors=custom_tech_colors, save_path="results/plots/stacked_area_metals_technologies/fig_sps_tech_eq")
fig_sps_tech_hh = plot_stacked_area(df_sps_damage, "Technology", "Total human health", "Human Health Damage by Technology", "DALY", custom_colors=custom_tech_colors, save_path="results/plots/stacked_area_metals_technologies/fig_sps_tech_hh")

# Contribution analysis

## Stacked barplot for MP contribution per year

In [ ]:
def generate_full_color_dict(impact_categories, custom_colors, colormap="tab20"):
    """
    Ensures all impact categories receive a distinct color.
    
    - Uses predefined colors for important categories.
    - Assigns unique colors to remaining categories from a colormap.

    Parameters:
    - impact_categories: List of all impact categories
    - custom_colors: Dictionary of manually defined colors for key categories
    - colormap: Matplotlib colormap to use for additional categories
    
    Returns:
    - A complete color dictionary for all impact categories
    """
    
    # Get a colormap with enough distinct colors
    cmap = cm.get_cmap(colormap, len(impact_categories))

    # Start with predefined colors
    full_color_dict = custom_colors.copy()

    # Assign unique colors to missing categories
    for i, cat in enumerate(impact_categories):
        if cat not in full_color_dict:
            full_color_dict[cat] = cmap(i)  # Assign a unique color

    return full_color_dict

def plot_stacked_bar(df, impact_columns, total_col, title, y_label="Percentage contribution (%)", 
                     color_palette="tab20", custom_colors=None, threshold=0.03, save_path="plots/stacked_bar"):
    """
    Generate a stacked bar plot showing the contribution of different midpoint indicators to the total impact.
    Contributors below a threshold (default: 5%) are grouped into "Other".

    Parameters:
    - df: DataFrame with data
    - impact_columns: List of midpoint impact indicators (e.g., EQ or HH categories)
    - total_col: Column containing the total impact for normalization (e.g., 'Total ecosystem quality' or 'Total human health')
    - title: Plot title
    - y_label: Label for Y-axis (default: "Percentage contribution (%)")
    - color_palette: Colormap name for automatic coloring (default: "tab20") if no custom colors are provided
    - custom_colors: Dictionary with predefined colors for certain categories
    - threshold: Minimum percentage contribution to remain separate; others are grouped into "Other"
    - save_path: Base filename for saving plots (without extension)
    
    Outputs:
    - Saves PDF and PNG versions of the figure
    """

    # Ensure directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Aggregate data by year
    df_grouped = df.groupby("Year")[impact_columns + [total_col]].sum()

    # Convert to percentage of total impact
    df_percent = df_grouped[impact_columns].div(df_grouped[total_col], axis=0) * 100

    # Identify small contributors to group into "Other"
    total_contributions = df_percent.mean()  # Average over time for consistency
    significant_categories = total_contributions[total_contributions >= (threshold * 100)].index.tolist()
    
    # Group small contributors into "Other"
    df_percent["Other"] = df_percent.drop(columns=significant_categories).sum(axis=1)
    df_percent = df_percent[significant_categories + ["Other"]]

    # Sort categories based on first year's contribution (largest to smallest)
    first_year = df_percent.index.min()
    sorted_categories = df_percent.loc[first_year].sort_values(ascending=False).index
    df_percent = df_percent[sorted_categories]

    # Assign colors: use custom colors if provided, else use a colormap
    if custom_colors:
        color_dict = {cat: custom_colors.get(cat, "gray") for cat in sorted_categories}
    else:
        color_dict = generate_full_color_dict(sorted_categories, {}, color_palette)

    # Ensure "Other" has a distinct neutral color
    color_dict["Other"] = "#2b2d42"  # Dark Gray for clarity

    # Generate plot
    fig, ax = plt.subplots(figsize=(7.2, 5))
    colors = [color_dict[col] for col in df_percent.columns]
    df_percent.plot(kind="bar", stacked=True, color=colors, ax=ax, width=0.8)

    # Formatting
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_xlabel("")
    ax.set_xticklabels(df_percent.index, rotation=360)

    # Legend formatting: Reduce size, place below if necessary
    #ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=6, frameon=False)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8, frameon=False)


    plt.tight_layout()

    # Save figures
    plt.savefig(f"{save_path}.pdf", format="pdf", dpi=600)
    plt.savefig(f"{save_path}.png", format="png", dpi=600)
    plt.show()


In [ ]:
# Define the directory for saving bar plots
barplot_dir = "results/plots/stacked_barplots_mp_contribution"
os.makedirs(barplot_dir, exist_ok=True)

In [ ]:
fig_sps_mp_eq = plot_stacked_bar(df_sps_damage, EQ, "Total ecosystem quality",
                 "Midpoint Contributions to Ecosystem Quality",
                 custom_colors=None,
                 save_path=f"{barplot_dir}/fig_sps_mp_eq")

In [ ]:
fig_sps_mp_hh = plot_stacked_bar(df_sps_damage, HH, "Total human health", 
                                 "Midpoint Contributions to Human Health", 
                                 custom_colors=None,
                                 save_path=f"{barplot_dir}/fig_sps_mp_hh")

In [ ]:
fig_nze_mp_eq = plot_stacked_bar(df_nze_damage, EQ, "Total ecosystem quality",
                 "Midpoint Contributions to Ecosystem Quality", 
                 custom_colors=None,
                 save_path=f"{barplot_dir}/fig_nze_mp_eq")

In [ ]:
fig_nze_mp_hh = plot_stacked_bar(df_nze_damage, HH, "Total human health", 
                                 "Midpoint Contributions to Human Health",
                                 custom_colors=None,
                                 save_path=f"{barplot_dir}/fig_nze_mp_hh")

## Barplot for MP contribution over the period

In [ ]:
def plot_cumulative_midpoint_contribution(df, impact_columns, total_col, title, y_label="Percentage contribution (%)", 
                                          color_palette="tab20", custom_colors=None, threshold=0.03, 
                                          save_path="plots/cumulative_midpoint"):
    """
    Generate a bar plot showing the cumulative contribution of different midpoint indicators to the total impact from 2022-2050.
    Bars represent the mean contribution, with a line showing min-max variation. Contributors below a threshold (default: 5%) 
    are grouped into "Other".

    Parameters:
    - df: DataFrame with data
    - impact_columns: List of midpoint impact indicators (e.g., EQ or HH categories)
    - total_col: Column containing the total impact for normalization (e.g., 'Total ecosystem quality' or 'Total human health')
    - title: Plot title
    - y_label: Label for Y-axis (default: "Percentage contribution (%)")
    - color_palette: Colormap name for automatic coloring (default: "tab20") if no custom colors are provided
    - custom_colors: Dictionary with predefined colors for certain categories
    - threshold: Minimum percentage contribution to remain separate; others are grouped into "Other"
    - save_path: Base filename for saving plots (without extension)
    
    Outputs:
    - Saves PDF and PNG versions of the figure
    """

    # Ensure directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Aggregate data over all years (2022-2050)
    df_grouped = df.groupby("Year")[impact_columns + [total_col]].sum()

    # Convert to percentage of total impact
    df_percent = df_grouped[impact_columns].div(df_grouped[total_col], axis=0) * 100

    # Compute mean, min, and max contributions over time
    mean_contributions = df_percent.mean()
    min_contributions = df_percent.min()
    max_contributions = df_percent.max()

    # Identify small contributors to group into "Other"
    significant_categories = mean_contributions[mean_contributions >= (threshold * 100)].index.tolist()
    
    # Group small contributors into "Other"
    df_percent["Other"] = df_percent.drop(columns=significant_categories).sum(axis=1)
    mean_contributions["Other"] = df_percent["Other"].mean()
    min_contributions["Other"] = df_percent["Other"].min()
    max_contributions["Other"] = df_percent["Other"].max()

    # Keep only significant contributors + "Other"
    mean_contributions = mean_contributions[significant_categories + ["Other"]]
    min_contributions = min_contributions[significant_categories + ["Other"]]
    max_contributions = max_contributions[significant_categories + ["Other"]]

    # Sort categories by mean contribution
    sorted_categories = mean_contributions.sort_values(ascending=False).index
    mean_contributions = mean_contributions[sorted_categories]
    min_contributions = min_contributions[sorted_categories]
    max_contributions = max_contributions[sorted_categories]

    # Assign colors: use custom colors if provided, else use a colormap
    if custom_colors:
        color_dict = {cat: custom_colors.get(cat, "gray") for cat in sorted_categories}
    else:
        cmap = cm.get_cmap(color_palette, len(sorted_categories))
        color_dict = {cat: cmap(i) for i, cat in enumerate(sorted_categories)}

    # Ensure "Other" has a distinct neutral color
    color_dict["Other"] = "#A9A9A9"  # Dark Gray for clarity

    # Generate plot
    fig, ax = plt.subplots(figsize=(9, 5))
    x_positions = np.arange(len(sorted_categories))
    colors = [color_dict[col] for col in sorted_categories]

    # Plot bars (mean values)
    bars = ax.bar(x_positions, mean_contributions, color=colors, alpha=0.8)

    # Add min-max variation as a line
    ax.errorbar(x_positions, mean_contributions, 
                yerr=[mean_contributions - min_contributions, max_contributions - mean_contributions], 
                fmt='none', ecolor='black', capsize=4, elinewidth=1)

    # Formatting
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(sorted_categories, rotation=90, ha="right", fontsize=8)

    # Save figures
    plt.tight_layout()
    plt.savefig(f"{save_path}.pdf", format="pdf", dpi=600)
    plt.savefig(f"{save_path}.png", format="png", dpi=600)
    plt.show()

In [ ]:
fig_sps_mp_eq_cum = plot_cumulative_midpoint_contribution(df_sps_damage, EQ, "Total ecosystem quality",
                                      "Cumulative Midpoint Contributions (2022-2050) - Ecosystem Quality",
                                      save_path="results/plots/barplots_mp_contribution_cumulative/fig_sps_mp_eq_cum")

fig_sps_mp_hh_cum = plot_cumulative_midpoint_contribution(df_sps_damage, HH, "Total human health",
                                      "Cumulative Midpoint Contributions (2022-2050) - Human Health",
                                      save_path="results/plots/barplots_mp_contribution_cumulative/fig_sps_mp_hh_cum")


In [ ]:
fig_nze_mp_eq_cum = plot_cumulative_midpoint_contribution(df_nze_damage, EQ, "Total ecosystem quality",
                                      "Cumulative Midpoint Contributions (2022-2050) - Ecosystem Quality",
                                      save_path="results/plots/barplots_mp_contribution_cumulative/fig_nze_mp_eq_cum")

fig_nze_mp_hh_cum = plot_cumulative_midpoint_contribution(df_nze_damage, HH, "Total human health",
                                      "Cumulative Midpoint Contributions (2022-2050) - Human Health",
                                      save_path="results/plots/barplots_mp_contribution_cumulative/fig_nze_mp_hh_cum")

# Sankey

In [ ]:
from plotting_functions import create_sankey

In [ ]:
from constants import agg_mapping_eq, agg_mapping_hh, metal_map

In [ ]:
# Define directory for saving Sankey diagrams
sankey_dir = "results/plots/sankey"
os.makedirs(sankey_dir, exist_ok=True)

In [ ]:
fig_sps_eq = create_sankey(
    df_sps_damage,
    total_col="Total ecosystem quality",
    impact_columns=EQ,
    #title="",
    save_path=f"{sankey_dir}/sps_eq",
    agg_mapping=agg_mapping_eq,
    metal_mapping=metal_map
)

fig_sps_hh = create_sankey(
    df_sps_damage,
    total_col="Total human health",
    impact_columns=HH,
    #title="",
    save_path=f"{sankey_dir}/sps_hh",
    agg_mapping=agg_mapping_hh,
    metal_mapping=metal_map
)


In [ ]:
fig_nze_eq = create_sankey(
    df_nze_damage,
    total_col="Total ecosystem quality",
    impact_columns=EQ,
    #title="",
    save_path=f"{sankey_dir}/nze_eq",
    agg_mapping=agg_mapping_eq,
    metal_mapping=metal_map
)

fig_nze_hh = create_sankey(
    df_nze_damage,
    total_col="Total human health",
    impact_columns=HH,
    #title="",
    save_path=f"{sankey_dir}/nze_hh",
    agg_mapping=agg_mapping_hh,
    metal_mapping=metal_map
)

# Benefits vs burden of the transition

In [ ]:
from plotting_functions import plot_lineplot_burden_comparison, plot_lineplot_burden_comparison_combined

In [ ]:
plot_lineplot_burden_comparison(df_comparison, save_path='results/plots/burden_comparison/lineplot')

In [ ]:
# Plotting the combined figure
plot_lineplot_burden_comparison_combined(df_comparison, save_path='results/plots/burden_comparison/lineplot_combined')

In [ ]:
from plotting_functions import plot_lineplot_logscale, plot_percentage_transition

In [ ]:
plot_lineplot_logscale(df_comparison, save_path='results/plots/burden_comparison/lineplot_logscale')

In [ ]:
plot_percentage_transition(df_comparison, save_path='results/plots/burden_comparison/percentage')

# Energy transition vs rest of the economy

In [ ]:
df_etm_roe = pd.read_csv(r'results/energy_transition_vs_roe.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.lines import Line2D
import math

# ——— Global Matplotlib parameters (Joule style) ———
rcParams['pdf.fonttype']   = 42
rcParams['ps.fonttype']    = 42
rcParams['font.family']    = 'Arial'
rcParams['font.size']      = 10
rcParams['axes.labelsize'] = 10
rcParams['legend.fontsize']= 9
rcParams['xtick.labelsize']= 9
rcParams['ytick.labelsize']= 9
rcParams['axes.titlesize'] = 12

def plot_all_metals_subplots(
    df,
    ncols=4,
    energy_color='#31a354',
    rest_color='#de2d26',
    save_path=None,
    show=False
):
    """
    Multi-panel line plot with shared legend placed in the empty bottom-right subplot.
    """
    metals = df['Metal'].unique()
    n = len(metals)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3), squeeze=False)

    # Plot each metal
    for idx, metal in enumerate(metals):
        r, c = divmod(idx, ncols)
        ax = axes[r][c]
        sub = df[df['Metal'] == metal]
        ax.plot(sub['Year'], sub['Energy transition (kt)'], marker='o',
                color=energy_color)
        ax.plot(sub['Year'], sub['Rest of the economy (kt)'], marker='s',
                color=rest_color)
        ax.set_title(metal)
        ax.set_xlabel('Year')
        ax.set_ylabel('Demand (kt)')

    # Prepare legend handles
    handles = [
        Line2D([0], [0], color=energy_color, marker='o', label='Energy transition'),
        Line2D([0], [0], color=rest_color, marker='s', label='Rest of economy')
    ]

    # Hide unused subplots and place legend in the first empty spot
    for idx in range(n, nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r][c].axis('off')
        # place legend in that first empty subplot
        #axes[r][c].legend(handles=handles, loc='lower right', frameon=False)
        
        # after — framed and larger
        legend = axes[r][c].legend(
    handles=handles,
    loc='lower right',
    frameon=True,               # turn the box on
    edgecolor='black',          # box border color
    facecolor='white',          # box fill
    fontsize=12,                # larger text
    markerscale=1.5,            # bigger markers
    handlelength=2.0,           # longer line samples
    borderpad=1.0               # space between box and content
)
        legend.get_frame().set_linewidth(1.0)  # thicker frame line
        break  # only use the first empty spot

    plt.tight_layout()

    # Save both PDF and PNG at high resolution
    if save_path:
        fig.savefig(f"{save_path}.pdf", format='pdf', dpi=600)
        fig.savefig(f"{save_path}.png", format='png', dpi=600)

    if show:
        plt.show()

    plt.close()

In [ ]:
plot_all_metals_subplots(
    df_etm_roe,
    ncols=4,
    save_path='results/plots/et_vs_roe/combined_subplots')